# 15 — Pull Freedom House (Freedom in the World)

**Source:** Freedom House — *Freedom in the World* annual report  
**Coverage:** ~210 countries and territories, 1972–present, annual  
**Access:** Direct CSV/Excel download from Freedom House website (no API key required)

## What this notebook does

1. Downloads the Freedom in the World aggregate scores dataset from Freedom House
2. Selects and renames the variables used in the instability prediction literature
3. Derives key binary labels used as dependent variables in the synthesis document
4. Writes a country-year parquet to ADLS

## Variables acquired

| Variable | Type | Description | Use |
|---|---|---|---|
| `fh_pr` | integer 1–7 | Political Rights score (1=most free) | Predictor + label |
| `fh_cl` | integer 1–7 | Civil Liberties score (1=most free) | Predictor + label |
| `fh_status` | categorical | Free / Partly Free / Not Free | Label for status change |
| `fh_total` | integer 0–100 | Aggregate freedom score (PR+CL combined) | Predictor |
| `fh_status_decline` | binary | 1 if FH status worsened vs. prior year | **Dependent variable** |
| `fh_pr_decline` | binary | 1 if PR score increased by ≥1 (more restricted) | **Dependent variable** |

## Why this matters

Freedom House scores appear as **predictors** in the IMF WP/21/291 unrest model and
the Harff 2003 genocide model, and as **dependent variables** in democratic backsliding
studies. `fh_status_decline` (downgrade from Free→Partly Free or Partly Free→Not Free)
is the operationalisation used in the regime change literature as a complement to V-Dem
ERT autocratization episodes.

## Output path on ADLS

```
raw/freedom_house/{RUN_DATE}/freedom_house_annual.parquet
```

Notebook `02/02_build_feature_matrix` already includes `"freedom_house": "raw/freedom_house"` in `RAW_PREFIXES`.

In [ ]:
%%capture
%pip install requests pandas openpyxl pyarrow azure-identity adlfs --quiet

In [ ]:
import io
import os
import warnings
from datetime import datetime

import pandas as pd
import numpy as np
import requests
import adlfs
from azure.identity import DefaultAzureCredential

warnings.filterwarnings('ignore')
RUN_DATE = datetime.utcnow().strftime('%Y%m%d')

In [ ]:
ADLS_ACCOUNT_NAME = os.environ['ADLS_ACCOUNT_NAME']
ADLS_CONTAINER    = os.getenv('ADLS_CONTAINER', 'data')

# Freedom House publishes the aggregate scores as an Excel file.
# URL pattern as of 2024/2025 — update year suffix if needed.
FH_URL = (
    'https://freedomhouse.org/sites/default/files/2025-03/'
    'All_data_FIW_2013-2025.xlsx'
)
# Fallback: the historical dataset covering 1972–2012 as separate file
FH_HISTORICAL_URL = (
    'https://freedomhouse.org/sites/default/files/2020-02/'
    'All_data_FIW_2013-2020.xlsx'
)
# Country-code crosswalk path (in-repo)
CROSSWALK_PATH = '../data/country_crosswalk.csv'

In [ ]:
credential = DefaultAzureCredential()
storage_options = {'account_name': ADLS_ACCOUNT_NAME, 'credential': credential}

def adls_path(subpath):
    return f'abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/{subpath}'

def write_parquet(df, subpath):
    df.to_parquet(adls_path(subpath), storage_options=storage_options,
                  index=False, engine='pyarrow')
    print(f'  Written {len(df):,} rows → {subpath}')

In [ ]:
def _download_fh(url, timeout=60):
    print(f'Downloading Freedom House data from {url} ...')
    resp = requests.get(url, timeout=timeout,
                        headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    xl = pd.ExcelFile(io.BytesIO(resp.content))
    print(f'  Sheets: {xl.sheet_names}')
    # FH workbooks typically have a 'FIW All Data' or 'Country Ratings' sheet
    data_sheet = next(
        (s for s in xl.sheet_names
         if any(k in s.lower() for k in ['fiw', 'country', 'data', 'scores'])),
        xl.sheet_names[0],
    )
    print(f'  Reading sheet: {data_sheet}')
    return pd.read_excel(xl, sheet_name=data_sheet, header=1)  # FH uses row 1 as header


df_raw = None
for url in [FH_URL, FH_HISTORICAL_URL]:
    try:
        df_raw = _download_fh(url)
        print(f'  Raw shape: {df_raw.shape}')
        break
    except Exception as exc:
        print(f'  Failed: {exc}')

if df_raw is None:
    raise RuntimeError(
        'Could not download Freedom House data. '
        'Download the Excel file manually from https://freedomhouse.org/reports/publication-archives '
        'and load with pd.read_excel().'
    )

In [ ]:
# Normalise the raw FH file into a tidy country-year panel.
# FH releases the data in two formats depending on year:
#   (a) Long: Country | Year | PR | CL | Status
#   (b) Wide: Country | Edition (= year) | ... sub-score columns

df = df_raw.copy()
df.columns = [str(c).strip().lower().replace(' ', '_') for c in df.columns]
print('Columns:', df.columns.tolist()[:20])

# Detect format
has_year = any(c in df.columns for c in ['year', 'edition', 'survey_year'])
year_col = next((c for c in ['year', 'edition', 'survey_year'] if c in df.columns), None)

# Map to standard names — FH column names vary by edition year
rename_map = {}
for src, tgt in [
    ('country/territory', 'country_name'), ('country', 'country_name'),
    ('pr', 'fh_pr'), ('political_rights', 'fh_pr'), ('political_rights_(1=most_free,_7=least_free)', 'fh_pr'),
    ('cl', 'fh_cl'), ('civil_liberties', 'fh_cl'), ('civil_liberties_(1=most_free,_7=least_free)', 'fh_cl'),
    ('status', 'fh_status'), ('freedom_status', 'fh_status'),
    ('total', 'fh_total'), ('total_score', 'fh_total'),
]:
    if src in df.columns:
        rename_map[src] = tgt
if year_col:
    rename_map[year_col] = 'year'

df = df.rename(columns=rename_map)
print('After rename:', [c for c in ['country_name', 'year', 'fh_pr', 'fh_cl', 'fh_status'] if c in df.columns])

In [ ]:
# Keep only country-level rows (FH also includes territories)
# and clean up
keep_cols = [c for c in ['country_name', 'year', 'fh_pr', 'fh_cl', 'fh_status', 'fh_total']
             if c in df.columns]
df = df[keep_cols].dropna(subset=['country_name', 'year']).copy()
df['year'] = pd.to_numeric(df['year'], errors='coerce').dropna().astype(int)
df = df.dropna(subset=['year'])

for col in ['fh_pr', 'fh_cl', 'fh_total']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Standardise status strings
if 'fh_status' in df.columns:
    df['fh_status'] = df['fh_status'].astype(str).str.strip().str.title()
    df['fh_status'] = df['fh_status'].replace({
        'F': 'Free', 'Pf': 'Partly Free', 'Nf': 'Not Free',
        'Free ': 'Free', 'Partly Free ': 'Partly Free',
    })

print(f'Rows after cleaning: {len(df):,}')
print(f'Years: {df["year"].min()}–{df["year"].max()}')
if 'fh_status' in df.columns:
    print('Status distribution:\n', df['fh_status'].value_counts().to_string())

In [ ]:
# Map country names to ISO3 using the crosswalk
from pathlib import Path

cw_path = Path(CROSSWALK_PATH)
if not cw_path.exists():
    cw_path = Path('data/country_crosswalk.csv')

df_cw = pd.read_csv(cw_path, dtype=str)
name_to_iso3 = dict(zip(df_cw['country_name_canonical'].str.lower(), df_cw['iso3']))

# FH uses its own country name spellings — add common alternates
fh_name_overrides = {
    'united states': 'USA', 'united kingdom': 'GBR', 'russia': 'RUS',
    'iran': 'IRN', 'south korea': 'KOR', 'north korea': 'PRK',
    'syria': 'SYR', 'vietnam': 'VNM', 'laos': 'LAO',
    'democratic republic of congo': 'COD', 'dr congo': 'COD',
    'republic of congo': 'COG', 'congo, republic of': 'COG',
    'congo, democratic republic of': 'COD',
    'ivory coast': 'CIV', "cote d'ivoire": 'CIV',
    'bolivia': 'BOL', 'venezuela': 'VEN', 'tanzania': 'TZA',
    'gambia, the': 'GMB', 'bahamas, the': 'BHS',
}

def _name_to_iso3(name):
    n = str(name).strip().lower()
    return fh_name_overrides.get(n) or name_to_iso3.get(n)

df['iso3'] = df['country_name'].apply(_name_to_iso3)

unmatched = df[df['iso3'].isna()]['country_name'].unique()
print(f'Unmatched countries ({len(unmatched)}): {sorted(unmatched)[:15]}')
df = df.dropna(subset=['iso3'])

In [ ]:
# Derive dependent-variable flags
df = df.sort_values(['iso3', 'year']).reset_index(drop=True)

# Status ordinal: Free=1, Partly Free=2, Not Free=3
status_ord = {'Free': 1, 'Partly Free': 2, 'Not Free': 3}
if 'fh_status' in df.columns:
    df['fh_status_ord'] = df['fh_status'].map(status_ord)
    # Decline = status worsened (higher ordinal) vs. prior year
    df['fh_status_prev'] = df.groupby('iso3')['fh_status_ord'].shift(1)
    df['fh_status_decline'] = (
        (df['fh_status_ord'] > df['fh_status_prev']).astype('Int64')
    )
    df = df.drop(columns=['fh_status_prev'])

# PR decline = score increased by ≥1 (higher = less free)
if 'fh_pr' in df.columns:
    df['fh_pr_prev'] = df.groupby('iso3')['fh_pr'].shift(1)
    df['fh_pr_decline'] = ((df['fh_pr'] - df['fh_pr_prev']) >= 1).astype('Int64')
    df = df.drop(columns=['fh_pr_prev'])

final_cols = [c for c in
    ['iso3', 'year', 'fh_pr', 'fh_cl', 'fh_total', 'fh_status',
     'fh_status_ord', 'fh_status_decline', 'fh_pr_decline']
    if c in df.columns]
df = df[final_cols]

print(f'Final panel: {len(df):,} country-years, {df["iso3"].nunique()} countries')
if 'fh_status_decline' in df.columns:
    print(f'fh_status_decline base rate: {df["fh_status_decline"].mean():.2%}')
df.head()

In [ ]:
write_parquet(df, f'raw/freedom_house/{RUN_DATE}/freedom_house_annual.parquet')

print()
print('=' * 55)
print('Notebook 14d complete')
print('=' * 55)
print(f'  Rows      : {len(df):,}')
print(f'  Countries : {df["iso3"].nunique()}')
print(f'  Years     : {df["year"].min()}–{df["year"].max()}')
print()
print('Labels written:')
print('  fh_status_decline  — 1 if FH status category worsened vs. prior year')
print('  fh_pr_decline      — 1 if Political Rights score increased by ≥1')
print('Predictors: fh_pr, fh_cl, fh_total, fh_status_ord')
print()
print('Next: add  "freedom_house": "raw/freedom_house"  to RAW_PREFIXES in notebook 14.')